# Does semantic relatedness beat spatial proximity?

This notebook tests **the belief**, not an implementation of it.

## The belief

> Give an LLM two nouns and it scores how related they are. So: name the object under the
> gaze, ask the LLM which *other* objects relate to it, and those are the tokens to keep.

Stripped of machinery, that is one falsifiable sentence:

**semantic relatedness to the gaze object predicts what the answer needs, better than distance from the gaze does.**

Our best question-free predictor today is a gaze blob at 22% — pure geometry, "near the eye
matters". Trained FRM got 18.7% and *lost to that geometry*. The claim above says the reason is
that FRM never had semantics. We have never tested proximity against relatedness directly.

## The design

For every example, for every object that is **not** the gaze object:

```
target      damage   = drop in log P(gold answer) when that whole object is masked out
predictor A relatedness(gaze object name -> this object name)     LANGUAGE ONLY
predictor B -distance(gaze point -> this object centroid)         GEOMETRY ONLY
control   C size                                                  patches in the mask
```

Then: per-example Spearman of each predictor against damage, and a **paired** test of A vs B.

## Four traps this notebook is built to avoid

1. **Size confound.** Big masks damage more. A predictor that merely tracks area would look
   brilliant. Reported: partial correlation with size ranked out, plus a damage-per-patch target.
2. **Q/A confound.** We already measured that answer-only objects take 2.4x the damage of
   question objects. If relatedness quietly encodes "is this an answer noun", it wins for the
   wrong reason. Reported: correlation of relatedness with the Q/A flag, and the whole test
   re-run **within answer-only objects**.
3. **Name-frequency artifact.** Common nouns get high LM probability regardless of the gaze
   object. Killed by using **PMI** (conditional minus unconditional), and checked by a
   **shuffled control** that pairs each gaze object with another example's objects.
4. **Aspect squash.** One patch is ~80px wide but ~142px tall in the source frame. Token-grid
   distance is not scene distance. The geometry baseline gets the fair, real-pixel version.

## Two relatedness scores

* **LM-PMI** — the belief taken literally, using the VLM's own language model:
  `log P(b | "A photograph of a {a}. In the same photograph you can also see a")` minus the same
  with no `{a}`. This is the "ask an LLM how related two nouns are" operation, done properly.
* **Embedding cosine** — cosine between the two words' input embeddings. Cheaper, weaker,
  included so a null result cannot be blamed on one scoring choice.

## Cost

~1 grounding forward + ~6 ablation forwards per example, plus a handful of tiny text-only
forwards for PMI. Budget **15-25 min** on a T4 for ~120 examples. Nothing is read from a
previous notebook - the sink mask, the objects and the damages are all built here.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy nltk
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, glob, json, time, gc, re, itertools
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import spearmanr, wilcoxon, rankdata, mannwhitneyu

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

DATA_ROOT  = "/content/drive/MyDrive/wearvqa_gaze_only"
OUT        = "/content/drive/MyDrive/wearvqa_relatedness.pt"
SINKF      = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
MODEL_ID   = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
N_PER_TYPE = 12
MAX_OBJ, MIN_PATCHES = 8, 2
N_SINK_PROBE = 12
SEED = 0

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer

types = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
samples = []
for t in types:
    for jp in sorted(glob.glob(os.path.join(DATA_ROOT, t, "*.json")))[:N_PER_TYPE]:
        m = json.load(open(jp)); ip = jp[:-5] + ".jpg"
        if os.path.exists(ip) and "gaze" in m and m.get("response"):
            samples.append(dict(type=t, img_path=ip, question=m["question"],
                                answer=m["response"], gaze=m["gaze"]))
print(f"{len(types)} types | {len(samples)} samples")

import nltk
NLTK = True
for pkg in ("punkt", "punkt_tab", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
try:
    nltk.pos_tag(nltk.word_tokenize("a red dog runs"))
except Exception as e:
    NLTK = False
    print("nltk POS unavailable, falling back to content words:", e)
print("nltk POS tagging:", NLTK)

## 2. Machinery

Grounding, ablation, and the text-only language-model scorer that makes the relatedness number.

In [ ]:
def build_inputs(image, question, answer=""):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt-1:].sum())

@torch.no_grad()
def ground(inp):
    """post-softmax attention averaged over heads and layers -> [L_all, L_all]."""
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        model(**inp)
    finally:
        S._unpatch_eager_globals(patched)
    L = int(inp["input_ids"].shape[1])
    acc, n = None, 0
    for m in model.modules():
        p = getattr(m, "_post_attn", None)
        if p is not None and p.shape[-1] == L and p.shape[-2] == L:
            a = p[0].float().mean(0)
            acc = a if acc is None else acc + a; n += 1
        for at in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, at):
                delattr(m, at)
    return acc / max(n, 1)

GENERIC = {"image", "picture", "photo", "thing", "things", "side", "front", "back",
           "top", "bottom", "left", "right", "part", "color", "colour", "shape", "type",
           "kind", "number", "purpose", "way", "use", "something", "area", "place"}

def nouns_of(text):
    if NLTK:
        out = [w.lower() for w, t in nltk.pos_tag(nltk.word_tokenize(text))
               if t.startswith("NN") and len(w) > 2]
    else:
        out = [RS._clean_token(w).lower() for w in re.findall(r"[A-Za-z]+", text)
               if len(w) > 2 and not RS.is_stopword_token(w)]
    seen, res = set(), []
    for w in out:
        if w not in seen and w not in GENERIC:
            seen.add(w); res.append(w)
    return res

def token_span(word, pieces):
    text = "".join(pieces).lower()
    hits, start = [], 0
    while True:
        j = text.find(word, start)
        if j < 0:
            break
        hits.append((j, j + len(word))); start = j + 1
    idx, pos = [], 0
    for i, p in enumerate(pieces):
        a, b = pos, pos + len(p)
        if any(b > s and a < e for s, e in hits):
            idx.append(i)
        pos = b
    return idx

def masks_for(ids):
    iid = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    im = ids == iid
    tm = (ids != iid) & (ids != (pad if pad is not None else -10**9))
    return torch.nonzero(im).squeeze(-1), torch.nonzero(tm).squeeze(-1)

# ---------- the relatedness scorer: text-only, no image ----------
POS_T = "A photograph of a {a}. In the same photograph you can also see a"
NEG_T = "A photograph. In the photograph you can also see a"

def _find_text_lm(model):
    """fallback path if the multimodal forward refuses a text-only batch."""
    for path in ("model.text_model", "model.model.text_model", "model.language_model",
                 "language_model.model", "model.model"):
        m = model
        try:
            for p in path.split("."):
                m = getattr(m, p)
        except AttributeError:
            continue
        if hasattr(m, "embed_tokens") or isinstance(m, torch.nn.Module) and hasattr(m, "layers"):
            return m
    return None

_TXT = {"mode": None}

@torch.no_grad()
def _text_logits(ids):
    if _TXT["mode"] != "inner":
        try:
            out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            _TXT["mode"] = "top"
            return out.logits[0].float()
        except Exception as e:
            print("top-level text-only forward failed, using inner LM:", type(e).__name__)
            _TXT["mode"] = "inner"
            _TXT["lm"] = _find_text_lm(model)
            assert _TXT["lm"] is not None, "could not locate the text decoder"
    h = _TXT["lm"](input_ids=ids, attention_mask=torch.ones_like(ids))
    h = h[0] if isinstance(h, tuple) else h.last_hidden_state
    return model.lm_head(h)[0].float()

@torch.no_grad()
def lm_logprob(prefix, cont):
    """sum log P(cont tokens | prefix), text only, no pixel_values."""
    n = len(tokenizer(prefix)["input_ids"])
    ids = torch.tensor([tokenizer(prefix + " " + cont)["input_ids"]], device=device)
    lp = torch.log_softmax(_text_logits(ids)[:-1], dim=-1)
    return float(lp.gather(-1, ids[0, 1:, None]).squeeze(-1)[n-1:].sum())

_neg_cache, _pmi_cache = {}, {}

def pmi(a, b):
    """log P(b | photo contains a) - log P(b | photo).  Unconditional term cancels
    word frequency, which is trap 3."""
    key = (a, b)
    if key not in _pmi_cache:
        if b not in _neg_cache:
            _neg_cache[b] = lm_logprob(NEG_T, b)
        _pmi_cache[key] = lm_logprob(POS_T.format(a=a), b) - _neg_cache[b]
    return _pmi_cache[key]

_emb_cache = {}

@torch.no_grad()
def emb_vec(w):
    if w not in _emb_cache:
        ids = tokenizer(" " + w, add_special_tokens=False)["input_ids"]
        v = model.get_input_embeddings()(torch.tensor(ids, device=device)).float().mean(0)
        _emb_cache[w] = F.normalize(v, dim=-1).cpu()
    return _emb_cache[w]

def emb_cos(a, b):
    return float(emb_vec(a) @ emb_vec(b))

print("sanity check on the relatedness scorer (higher = more related):")
for a, b in [("refrigerator", "milk"), ("refrigerator", "ceiling"),
             ("laptop", "keyboard"), ("laptop", "banana"),
             ("shelf", "price"), ("shelf", "cloud")]:
    print(f"   pmi({a:<13}, {b:<9}) = {pmi(a,b):+.2f}    cos = {emb_cos(a,b):+.3f}")
print("\nIf the related pairs are not clearly above the unrelated pairs, the scorer is broken")
print("and everything below is meaningless. Check this before reading any result.")

## 3. Sink mask (built here, not loaded)

In [ ]:
if os.path.exists(SINKF):
    blob_ = torch.load(SINKF, weights_only=False)
    sinks, sink_score = blob_["sink_mask"].bool(), blob_["sink_score"]
    print(f"loaded sink mask: {torch.nonzero(sinks).squeeze(-1).tolist()}")
else:
    step = max(1, len(samples) // N_SINK_PROBE)
    sc = []
    for s in samples[::step][:N_SINK_PROBE]:
        o = S.make_smolvlm_output(image=S.load_image(s["img_path"]), question=s["question"])
        if o is None:
            continue
        sc.append(VS.sink_scores(o.post_softmax, o.image_token_mask, o.text_token_mask,
                                 is_post_softmax=True))
        del o; gc.collect(); torch.cuda.empty_cache()
    sink_score = VS.aggregate_sink_scores(sc)
    assert float(sink_score.median()) > 0, "sink scores collapsed to zero - see the 2b87c7c fix"
    sinks = VS.detect_sinks(sink_score)
    torch.save({"model": MODEL_ID, "L_v": sink_score.numel(), "sink_mask": sinks,
                "sink_score": sink_score, "n_probe": len(sc)}, SINKF)
    print(VS.sink_report(sink_score, sinks))

## 4. Objects, and which one the eye is on

Patches are assigned to a **named** object by text->vision attention at that word's token rows —
the localisation route Luo et al. measured at 56/75/74 recall against CLIP's 20/30. Two gates
run before any ablation GPU is spent.

In [ ]:
objs, t0 = [], time.time()
G = None
for i, s in enumerate(samples):
    img = S.load_image(s["img_path"])
    inp, n_prompt = build_inputs(img, s["question"], s["answer"])
    ids = inp["input_ids"][0].cpu()
    vpos, tpos = masks_for(ids)
    L_v = len(vpos)
    if G is None:
        G = int(round(math.sqrt(L_v)))

    A = ground(inp)
    toks = tokenizer.convert_ids_to_tokens(ids[tpos].tolist())
    pieces = [RS._detok_piece(t) for t in toks]

    qn, an = nouns_of(s["question"]), nouns_of(s["answer"])
    names = (qn + [w for w in an if w not in qn])[:MAX_OBJ]

    maps, kept, src = [], [], []
    for w in names:
        rows = token_span(w, pieces)
        if not rows:
            continue
        m = A[tpos[rows]][:, vpos].mean(0)
        m = m / m.sum().clamp_min(1e-9)
        maps.append(m); kept.append(w); src.append("Q" if w in qn else "A")

    del A, inp; gc.collect(); torch.cuda.empty_cache()
    if len(maps) < 3:                      # need a gaze object + >=2 others to rank
        objs.append(None); continue

    M = torch.stack(maps, 0)
    lab = M.argmax(0)
    gp = (min(G-1, int(s["gaze"]["y_norm"]*G)) * G + min(G-1, int(s["gaze"]["x_norm"]*G)))
    objs.append(dict(idx=i, names=kept, src=src, M=M, lab=lab, gp=gp,
                     gaze_obj=int(lab[gp]), W=img.size[0], H=img.size[1]))
    if (i+1) % 25 == 0:
        print(f"  grounded {i+1}/{len(samples)}  ({(time.time()-t0)/60:.1f} min)")

good = [o for o in objs if o is not None]
print(f"\nusable: {len(good)}/{len(samples)}   objects/image mean "
      f"{np.mean([len(o['names']) for o in good]):.1f}")

cors = []
for o in good:
    Mn = F.normalize(o["M"] - o["M"].mean(1, keepdim=True), dim=1)
    C = (Mn @ Mn.T).numpy()
    cors += [C[a, b] for a in range(len(C)) for b in range(a+1, len(C))]
print(f"GATE  mean pairwise map correlation {np.mean(cors):.3f}  (near 1.0 = no discrimination)")

gq = np.mean([o["src"][o["gaze_obj"]] == "Q" for o in good])
print(f"      gaze object is a QUESTION noun {gq:.0%} of the time")
print("      (your chain assumes the eye lands on the question object; this is that number)")

In [ ]:
from PIL import Image as _I
fig, ax = plt.subplots(3, 4, figsize=(15, 11))
for a, o in zip(ax.ravel(), good[:12]):
    s = samples[o["idx"]]; img = S.load_image(s["img_path"]); W, H = img.size
    seg = _I.fromarray((o["lab"].reshape(G, G).numpy() *
                        (255 // max(len(o["names"]), 1))).astype("uint8")).resize((W, H), _I.NEAREST)
    a.imshow(img); a.imshow(np.array(seg), cmap="tab10", alpha=0.55)
    a.scatter([s["gaze"]["x_norm"]*W], [s["gaze"]["y_norm"]*H],
              marker="x", s=140, c="lime", linewidths=3)
    ttl = " / ".join(("*" if j == o["gaze_obj"] else "") + f"{n}[{t}]"
                     for j, (n, t) in enumerate(zip(o["names"], o["src"])))
    a.set_title(ttl[:74], fontsize=6); a.axis("off")
plt.tight_layout(); plt.show()
print("* marks the object the gaze fell on. Green x is the raw gaze point.")

## 5. Measure the damage

Mask out one whole object at a time and record the drop in log P(gold answer). This is the
target both predictors have to beat each other on.

In [ ]:
if os.path.exists(OUT):
    res = torch.load(OUT, weights_only=False)
    print(f"resuming from {len(res)}")
else:
    res = []
done = {r["idx"] for r in res}

t0 = time.time()
for o in good:
    if o["idx"] in done:
        continue
    s = samples[o["idx"]]
    inp, n_prompt = build_inputs(S.load_image(s["img_path"]), s["question"], s["answer"])
    ids = inp["input_ids"][0].cpu()
    vpos, _ = masks_for(ids)
    L_v = len(vpos)
    base = answer_logprob(inp, n_prompt)

    n_obj = len(o["names"])
    dmg = torch.full((n_obj,), float("nan"))
    sz  = torch.zeros(n_obj)
    cx  = torch.zeros(n_obj); cy = torch.zeros(n_obj)
    for r in range(n_obj):
        sel = ((o["lab"] == r) & (~sinks[:L_v])).nonzero().squeeze(-1)
        sz[r] = len(sel)
        if len(sel) < MIN_PATCHES:
            continue
        rows = (sel // G).float(); cols = (sel % G).float()
        cx[r] = float((cols.mean() + .5) / G)      # normalised centroid
        cy[r] = float((rows.mean() + .5) / G)
        am = inp["attention_mask"].clone(); am[0, vpos[sel]] = 0
        dmg[r] = base - answer_logprob(inp, n_prompt, am)

    res.append(dict(idx=o["idx"], base=base, dmg=dmg, sz=sz, cx=cx, cy=cy,
                    names=o["names"], src=o["src"], gaze_obj=o["gaze_obj"],
                    gp=o["gp"], W=o["W"], H=o["H"], lab=o["lab"]))
    del inp; gc.collect(); torch.cuda.empty_cache()
    if len(res) % 20 == 0:
        torch.save(res, OUT); print(f"  ablated {len(res)}/{len(good)}  ({(time.time()-t0)/60:.1f} min)")

torch.save(res, OUT)
print(f"done in {(time.time()-t0)/60:.1f} min -> {OUT}")

## 6. Build the table: relatedness, distance, size, damage

One row per (gaze object -> other object) pair. Distance is in **real source pixels**, not grid
units, because a patch is ~80px wide and ~142px tall — the geometry baseline gets the fair
version of itself.

In [ ]:
rng = np.random.default_rng(SEED)
rows, per_ex = [], []

# shuffled control: pair each example's objects with a DIFFERENT example's gaze object
shuf = rng.permutation(len(res))

for e, r in enumerate(res):
    g = r["gaze_obj"]
    ga = r["names"][g]
    fake_r = res[int(shuf[e])]
    fake_a = fake_r["names"][fake_r["gaze_obj"]]
    gx = samples[r["idx"]]["gaze"]["x_norm"] * r["W"]
    gy = samples[r["idx"]]["gaze"]["y_norm"] * r["H"]

    loc = []
    for j in range(len(r["names"])):
        if j == g or not math.isfinite(float(r["dmg"][j])):
            continue
        b = r["names"][j]
        px, py = float(r["cx"][j]) * r["W"], float(r["cy"][j]) * r["H"]
        dist = math.hypot(px - gx, py - gy)
        loc.append(dict(ex=e, a=ga, b=b, src=r["src"][j],
                        dmg=float(r["dmg"][j]), size=float(r["sz"][j]),
                        pmi=pmi(ga, b), cos=emb_cos(ga, b),
                        pmi_shuf=pmi(fake_a, b),
                        negdist=-dist, dist=dist))
    if len(loc) >= 3:
        rows += loc; per_ex.append(loc)

print(f"{len(rows)} (gaze object -> other object) pairs from {len(per_ex)} examples")
print(f"mean others per example {np.mean([len(p) for p in per_ex]):.1f}")

D  = {k: np.array([x[k] for x in rows], float)
      for k in ("dmg", "size", "pmi", "cos", "pmi_shuf", "negdist", "dist")}
D["dmg_per_patch"] = D["dmg"] / np.maximum(D["size"], 1)
SRC = np.array([x["src"] for x in rows])

print(f"\nCONFOUND CHECK (trap 2)")
print(f"  relatedness vs 'is an answer noun' : rho {spearmanr(D['pmi'], (SRC=='A').astype(float))[0]:+.3f}")
print(f"  relatedness vs size                : rho {spearmanr(D['pmi'], D['size'])[0]:+.3f}")
print(f"  distance     vs size               : rho {spearmanr(D['negdist'], D['size'])[0]:+.3f}")
print(f"  damage       vs size               : rho {spearmanr(D['dmg'], D['size'])[0]:+.3f}")

## 7. THE TEST

In [ ]:
PREDS = [("relatedness (LM-PMI)", "pmi"),
         ("relatedness (emb cos)", "cos"),
         ("-distance from gaze",  "negdist"),
         ("size",                 "size"),
         ("CTRL shuffled PMI",    "pmi_shuf")]

def per_example_rho(key, target="dmg"):
    out = []
    for p in per_ex:
        x = np.array([q[key] for q in p], float)
        y = np.array([q[target] for q in p], float)
        if len(set(x.tolist())) < 2 or len(set(y.tolist())) < 2:
            continue
        rho = spearmanr(x, y)[0]
        if np.isfinite(rho):
            out.append(rho)
    return np.array(out)

def prec_at_1(key):
    hit = 0
    for p in per_ex:
        x = np.array([q[key] for q in p]); y = np.array([q["dmg"] for q in p])
        hit += int(int(x.argmax()) == int(y.argmax()))
    return hit / len(per_ex)

def partial_rho(key, ctrl="size", target="dmg"):
    rx, ry, rz = rankdata(D[key]), rankdata(D[target]), rankdata(D[ctrl])
    B = np.c_[np.ones_like(rz), rz]
    ex = rx - B @ np.linalg.lstsq(B, rx, rcond=None)[0]
    ey = ry - B @ np.linalg.lstsq(B, ry, rcond=None)[0]
    return float(np.corrcoef(ex, ey)[0, 1])

chance = np.mean([1/len(p) for p in per_ex])
print(f"{'predictor':<24} {'per-ex rho':>11} {'p vs 0':>9} {'pooled':>8} "
      f"{'-size':>7} {'/patch':>7} {'p@1':>6}")
print("-" * 80)
store = {}
for label, key in PREDS:
    r = per_example_rho(key); store[key] = r
    p0 = wilcoxon(r)[1] if len(r) > 5 and np.any(r != 0) else float("nan")
    print(f"{label:<24} {r.mean():>+11.3f} {p0:>9.3g} "
          f"{spearmanr(D[key], D['dmg'])[0]:>+8.3f} {partial_rho(key):>+7.3f} "
          f"{spearmanr(D[key], D['dmg_per_patch'])[0]:>+7.3f} {prec_at_1(key):>6.0%}")
print("-" * 80)
print(f"p@1 chance = {chance:.0%}   n_examples = {len(per_ex)}   n_pairs = {len(rows)}")
print("\nper-ex rho = mean within-example Spearman vs damage (the honest one)")
print("-size      = partial correlation with mask size ranked out  (trap 1)")
print("/patch     = correlation against damage per patch instead of total")

### 7b. Head to head — the actual question

In [ ]:
def head_to_head(k1, k2, l1, l2):
    common = min(len(store[k1]), len(store[k2]))
    a, b = store[k1][:common], store[k2][:common]
    d = a - b
    p = wilcoxon(a, b)[1] if np.any(d != 0) else float("nan")
    print(f"{l1} {a.mean():+.3f}   vs   {l2} {b.mean():+.3f}")
    print(f"   difference {d.mean():+.3f}   paired p {p:.3g}   "
          f"{l1} wins {(d>0).mean():.0%} of examples\n")

print("=== RELATEDNESS vs GEOMETRY ===\n")
head_to_head("pmi", "negdist", "relatedness", "-distance")
head_to_head("cos", "negdist", "emb cosine ", "-distance")
print("=== IS RELATEDNESS ABOVE ITS OWN NULL? (trap 3) ===\n")
head_to_head("pmi", "pmi_shuf", "relatedness", "shuffled   ")

print("=== WITHIN ANSWER-ONLY OBJECTS (trap 2) ===\n")
sub = [[q for q in p if q["src"] == "A"] for p in per_ex]
sub = [p for p in sub if len(p) >= 3]
if len(sub) >= 8:
    def sub_rho(key):
        o = []
        for p in sub:
            x = np.array([q[key] for q in p], float); y = np.array([q["dmg"] for q in p], float)
            if len(set(x.tolist())) > 1 and len(set(y.tolist())) > 1:
                v = spearmanr(x, y)[0]
                if np.isfinite(v):
                    o.append(v)
        return np.array(o)
    rp, rd = sub_rho("pmi"), sub_rho("negdist")
    n = min(len(rp), len(rd))
    print(f"n={len(sub)} examples with >=3 answer-only objects")
    print(f"   relatedness {rp.mean():+.3f}    -distance {rd.mean():+.3f}")
    if n > 5 and np.any(rp[:n] != rd[:n]):
        print(f"   paired p {wilcoxon(rp[:n], rd[:n])[1]:.3g}")
    print("   if relatedness only won overall but ties here, it was reading the Q/A flag.")
else:
    print(f"only {len(sub)} examples have >=3 answer-only objects - subset test skipped")

### 7c. Where does the gaze actually sit?

Your chain predicts: question objects sit **near** the gaze, answer objects sit **further away**.
If both sit equally close, the whole premise of "look here, so attend over there" is unnecessary
— a blob around the gaze already covers everything.

In [ ]:
dq = D["dist"][SRC == "Q"]; da = D["dist"][SRC == "A"]
print(f"distance from gaze to QUESTION objects  {dq.mean():7.1f} px   (n={len(dq)})")
print(f"distance from gaze to ANSWER-only objs  {da.mean():7.1f} px   (n={len(da)})")
if len(dq) > 5 and len(da) > 5:
    print(f"Mann-Whitney (answer further) p {mannwhitneyu(da, dq, alternative='greater')[1]:.3g}")
print(f"\ndamage on QUESTION objects  {D['dmg'][SRC=='Q'].mean():.3f}")
print(f"damage on ANSWER-only objs  {D['dmg'][SRC=='A'].mean():.3f}")
print(f"size  QUESTION {D['size'][SRC=='Q'].mean():.1f} patches   "
      f"ANSWER-only {D['size'][SRC=='A'].mean():.1f} patches")
print("\nthe size line is the fair-Test-4 check: if answer objects are simply bigger,")
print("the 2.4x damage gap we saw earlier was area, not importance.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
zz = lambda v: (v - v.mean()) / (v.std() + 1e-9)
for a, (label, key) in zip(ax, PREDS[:3]):
    a.scatter(zz(D[key]), zz(D["dmg"]), s=12, alpha=.45,
              c=["tab:orange" if t == "A" else "tab:blue" for t in SRC])
    a.set_xlabel(label + "  (z)"); a.set_ylabel("damage (z)")
    a.set_title(f"{label}\npooled rho {spearmanr(D[key], D['dmg'])[0]:+.3f}", fontsize=10)
    a.axhline(0, lw=.5, c="k"); a.axvline(0, lw=.5, c="k")
plt.tight_layout(); plt.show()
print("blue = object named in the question, orange = named only in the answer")

fig, ax = plt.subplots(figsize=(7, 4))
lab = [l for l, _ in PREDS]
ax.bar(range(len(PREDS)), [store[k].mean() for _, k in PREDS],
       color=["tab:green", "tab:olive", "tab:red", "tab:gray", "lightgray"])
ax.axhline(0, c="k", lw=.8)
ax.set_xticks(range(len(PREDS))); ax.set_xticklabels(lab, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("mean within-example Spearman vs damage")
plt.tight_layout(); plt.show()

## 8. How to read this

The belief under test: **semantics beats geometry for deciding what to keep, given only the gaze.**

| outcome | reading |
|---|---|
| `relatedness > -distance`, paired p < 0.05, **and** it survives the size partial, the shuffled control and the answer-only subset | the intuition is right at the source. FRM should be rebuilt on language-space relatedness rather than a learned dot product on embeddings, and the engineering problems (naming the gaze patch, finer grid, SAM) become worth solving. |
| `relatedness ~ -distance` | semantics adds nothing geometry did not already have. The gaze blob at 22% is then close to the real question-free ceiling and FRM has no headroom to recover. |
| `relatedness > -distance` but it dies in the size partial or the answer-only subset | it was reading area or the Q/A flag, not relatedness. Same verdict as a tie. |
| `CTRL shuffled PMI ~ relatedness` | the scorer is measuring how photographable a noun is, not how it relates to the gaze object. Nothing was tested; fix the scorer first. |

Two numbers to check **before** trusting any of it: the relatedness sanity pairs printed in
section 2, and the map-correlation gate in section 4. If either is bad, the rest is noise about
noise — which is the exact failure mode that has cost us the last ten notebooks.